<a href="https://colab.research.google.com/github/spirosChv/neuro208-tutorials/blob/main/7_NeuroAI/Notebook4_surrogate_gradients.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Dead Neuron Problem and Surrogate Gradients

We've seen how a CUBA-LIF neuron works and how a feedforward SNN is constructed. But how do we **train** such a network with gradient descent? Spiking neurons pose a fundamental challenge: their output is **non-differentiable**.

This notebook explains:
1. Why SNNs behave like **recurrent networks**
2. How the **chain rule** breaks down at the spiking nonlinearity
3. The **dead neuron problem** — why gradients vanish
4. **Surrogate gradients** — the practical solution
5. How to implement a surrogate gradient in **PyTorch**

In [ ]:
!pip install mplcyberpunk -q

In [ ]:
import torch
import matplotlib.pyplot as plt
import mplcyberpunk

## 1. SNNs Are Recurrent Networks

Recall the LIF update equation from the previous notebooks:

$$V_{mem}[t] = \alpha_{mem} \cdot V_{mem}[t-1] + I_{in}[t] - R[t]$$

The membrane potential at time $t$ depends on its own value at time $t-1$. This is a **recurrence** — the neuron feeds its own state back to itself across timesteps. When we unroll this across time, the computational graph looks exactly like a Recurrent Neural Network (RNN):

<center>
<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/examples/tutorial5/unrolled_2.png?raw=true' width="700">
</center>

*Figure from [snnTorch Tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html) by J. Eshraghian.*

This means that to train an SNN, we need **Backpropagation Through Time (BPTT)** — the same algorithm used to train RNNs and LSTMs. The gradient must flow backward through every timestep of the simulation. Fortunately, PyTorch's autograd handles this automatically — we just need to make sure each operation in the forward pass is differentiable.

## 2. Applying the Chain Rule

Consider a single timestep of the forward pass:

<center>
<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/examples/tutorial5/non-diff.png?raw=true' width="350">
</center>

*Figure from [snnTorch Tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html) by J. Eshraghian.*

To update a weight $W$ using gradient descent, we need the gradient of the loss with respect to $W$. Applying the chain rule:

$$\frac{\partial \mathcal{L}}{\partial W} =
\frac{\partial \mathcal{L}}{\partial S}
\underbrace{\frac{\partial S}{\partial U}}_{\text{problem!}}
\frac{\partial U}{\partial I}
\frac{\partial I}{\partial W}$$

Most of these terms are straightforward:
- $\partial I / \partial W = X$ (the input)
- $\partial U / \partial I = 1$ (linear relationship)
- $\partial \mathcal{L} / \partial S$ depends on the loss function (e.g., cross-entropy)

But what about $\partial S / \partial U$? This is where things break down.

## 3. The Dead Neuron Problem

The spike generation function is the **Heaviside step function**:

$$S = \Theta(U - U_{thr}) = \begin{cases} 1, & \text{if } U > U_{thr} \\ 0, & \text{otherwise} \end{cases}$$

<center>
<img src='https://github.com/jeshraghian/snntorch/blob/master/docs/_static/img/examples/tutorial3/3_2_spike_descrip.png?raw=true' width="500">
</center>

*Figure from [snnTorch Tutorial 3](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_3.html) by J. Eshraghian.*

The derivative of the Heaviside function is the **Dirac delta**:

$$\frac{\partial S}{\partial U} = \delta(U - U_{thr}) = \begin{cases} +\infty, & \text{if } U = U_{thr} \\ 0, & \text{everywhere else} \end{cases}$$

This means:
- If the membrane is **below threshold**: gradient = 0 → no learning
- If the membrane is **above threshold**: gradient = 0 → no learning  
- If the membrane is **exactly at threshold**: gradient = $\infty$ → unstable

The gradient is **almost always zero**, killing all weight updates. This is the **dead neuron problem**.

Let's visualize this:

In [ ]:
U = torch.linspace(-2, 2, 1000)
threshold = 0.0  # shifted so threshold is at 0 for clarity

# Heaviside (forward pass)
heaviside = (U > threshold).float()

# Derivative of Heaviside = 0 everywhere (Dirac delta can't be plotted)
heaviside_grad = torch.zeros_like(U)

with plt.style.context("cyberpunk"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(U.numpy(), heaviside.numpy(), linewidth=2, color='#08F7FE')
    ax1.set_xlabel('$U - U_{thr}$', fontsize=12, color='white')
    ax1.set_ylabel('$S$', fontsize=12, color='white')
    ax1.set_title('Forward pass: Heaviside function', fontsize=12, color='white')
    ax1.tick_params(colors='white')
    mplcyberpunk.add_glow_effects(ax1)

    ax2.plot(U.numpy(), heaviside_grad.numpy(), linewidth=2, color='#FE53BB')
    ax2.set_xlabel('$U - U_{thr}$', fontsize=12, color='white')
    ax2.set_ylabel('$\\partial S / \\partial U$', fontsize=12, color='white')
    ax2.set_title('Backward pass: gradient is ZERO everywhere', fontsize=12, color='white')
    ax2.set_ylim(-0.1, 1.1)
    ax2.tick_params(colors='white')
    mplcyberpunk.add_glow_effects(ax2)

    fig.tight_layout()
    plt.show()

print("The gradient is zero everywhere → no weight updates → dead neurons!")

## 4. Surrogate Gradients: The Solution

The idea is simple:
- **Forward pass**: keep the Heaviside function as-is (spikes are binary)
- **Backward pass**: replace its derivative with a smooth function that has useful gradients

This substitute derivative is called a **surrogate gradient**. It may seem like a hack, but it turns out neural networks are remarkably robust to this kind of approximation.

### The Arctangent Surrogate

A popular choice (and what we use in our network) is the derivative of the arctangent function:

$$\frac{\partial \tilde{S}}{\partial U} = \frac{\alpha/2}{1 + \left(\frac{\pi}{2} \alpha (U - U_{thr})\right)^2}$$

where $\alpha$ controls the steepness. Higher $\alpha$ makes it closer to the true Dirac delta, lower $\alpha$ makes it smoother.

Let's compare the surrogate gradient to the (useless) true gradient:

In [ ]:
def atan_surrogate_grad(U, alpha):
    """Arctangent surrogate gradient."""
    return (alpha / 2) / (1.0 + (torch.pi / 2 * alpha * U).pow(2))


def atan_surrogate_forward(U, alpha):
    """Arctangent surrogate forward (smooth approximation of Heaviside)."""
    return 0.5 + (1 / torch.pi) * torch.atan(alpha * U)


alphas = [10.0, 5.0, 2.0]

with plt.style.context("cyberpunk"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Left: forward pass — Heaviside vs smooth ATan approximations
    ax1.plot(U.numpy(), heaviside.numpy(), linewidth=2.5, color='lightgray',
             linestyle='--', label='Heaviside (true)')
    for alpha in alphas:
        fwd = atan_surrogate_forward(U, alpha)
        ax1.plot(U.numpy(), fwd.numpy(), linewidth=1.5, label=f'ATan ($\\alpha$={int(alpha)})')
    ax1.set_xlabel('$U - U_{thr}$', fontsize=12, color='white')
    ax1.set_ylabel('$S$', fontsize=12, color='white')
    ax1.set_title('Forward pass: Heaviside vs smooth approximations', fontsize=13, color='white')
    ax1.tick_params(colors='white')
    ax1.legend(fontsize=11)
    mplcyberpunk.add_glow_effects(ax1)


    # Right: surrogate gradients for different alpha
    ax2.plot(U.numpy(), heaviside_grad.numpy(), linewidth=2, color='lightgray',
             linestyle='--', label='True gradient (dead!)')
    for alpha in alphas:
        grad = atan_surrogate_grad(U, alpha)
        ax2.plot(U.numpy(), grad.numpy(), linewidth=1.5, label=f'ATan ($\\alpha$={int(alpha)})')
    ax2.set_xlabel('$U - U_{thr}$', fontsize=12, color='white')
    ax2.set_ylabel('$\\partial \\tilde{S} / \\partial U$', fontsize=12, color='white')
    ax2.set_title('Backward pass: surrogate vs true gradient', fontsize=13, color='white')
    ax2.tick_params(colors='white')
    ax2.legend(fontsize=10)
    mplcyberpunk.add_glow_effects(ax2)

    fig.tight_layout()
    plt.show()

The surrogate gradient is:
- **Largest near the threshold** — where a small change in membrane potential is most likely to change the spiking outcome
- **Smoothly decays to zero** far from the threshold — neurons far from firing contribute less to the gradient
- **Controlled by $\alpha$** — higher values make the surrogate sharper (closer to the true derivative), lower values spread the gradient more broadly

## 5. Our PyTorch Implementation: `ATanSurrogate`

In PyTorch, we implement the surrogate gradient using a custom `autograd.Function`. This lets us define:
- A **forward pass** that applies the Heaviside step function (binary spikes)
- A **backward pass** that substitutes the arctangent surrogate gradient

In [ ]:
class ATanSurrogate(torch.autograd.Function):
    """
    Forward: Heaviside step function (binary spikes)
    Backward: Arctangent surrogate gradient
    """
    @staticmethod
    def forward(ctx, x, alpha=2.0):
        ctx.save_for_backward(x)
        ctx.alpha = alpha
        return (x > 0).float()  # Heaviside

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        alpha = ctx.alpha
        # Surrogate gradient: smooth, bell-shaped curve centered at threshold
        grad_x = (alpha / 2) / (1.0 + (torch.pi / 2 * alpha * x).pow(2))
        return grad_output * grad_x, None

Let's verify that it works — the forward pass should produce binary spikes, while the backward pass should produce smooth gradients:

In [ ]:
# Test input: membrane potential relative to threshold
x = torch.linspace(-2, 2, 1000, requires_grad=True)

# Forward pass: binary spikes
spikes = ATanSurrogate.apply(x, 2.0)

# Backward pass: compute surrogate gradient
spikes.sum().backward()

with plt.style.context("cyberpunk"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(x.detach().numpy(), spikes.detach().numpy(), linewidth=2, color='#08F7FE')
    ax1.set_xlabel('$U - U_{thr}$', fontsize=12, color='white')
    ax1.set_ylabel('Spike output', fontsize=12, color='white')
    ax1.set_title('Forward: Heaviside (binary)', fontsize=12, color='white')
    ax1.tick_params(colors='white')
    mplcyberpunk.add_glow_effects(ax1)

    ax2.plot(x.detach().numpy(), x.grad.numpy(), linewidth=2, color='#FE53BB')
    ax2.set_xlabel('$U - U_{thr}$', fontsize=12, color='white')
    ax2.set_ylabel('Gradient', fontsize=12, color='white')
    ax2.set_title('Backward: ATan surrogate gradient', fontsize=12, color='white')
    ax2.tick_params(colors='white')
    mplcyberpunk.add_glow_effects(ax2)

    fig.tight_layout()
    plt.show()

print("Forward: binary spikes (0 or 1)")
print("Backward: smooth, non-zero gradients that enable learning!")

## Summary

| Problem | Cause | Solution |
|---|---|---|
| SNNs are recurrent | Membrane potential depends on previous timestep | PyTorch autograd handles BPTT automatically |
| **Dead neuron problem** | Heaviside derivative is 0 almost everywhere | **Surrogate gradient**: smooth substitute in backward pass |

The key insight is that we **don't need the exact gradient** — a smooth approximation that points in roughly the right direction is enough for gradient descent to work. This is why surrogate gradients have become the dominant method for training SNNs.

In the next notebook, we will use `ATanSurrogate` inside our CUBA-LIF network and train it on MNIST with gradient descent.

---

### Further reading

- E. O. Neftci, H. Mostafa, F. Zenke, ["Surrogate Gradient Learning in Spiking Neural Networks"](https://ieeexplore.ieee.org/document/8891809), IEEE Signal Processing Magazine, 2019.
- F. Zenke and S. Ganguli, ["SuperSpike: Supervised Learning in Multilayer Spiking Neural Networks"](https://direct.mit.edu/neco/article/30/6/1514/8378), Neural Computation, 2018.
- [snnTorch tutorials](https://snntorch.readthedocs.io/en/latest/tutorials/index.html) by J. Eshraghian.

---

*This notebook is adapted from [snnTorch Tutorial 5](https://snntorch.readthedocs.io/en/latest/tutorials/tutorial_5.html) by Jason K. Eshraghian. Figures are used with credit to the [snnTorch project](https://github.com/jeshraghian/snntorch). If you find these resources useful, please consider citing:*

> Jason K. Eshraghian, Max Ward, Emre Neftci, Xinxin Wang, Gregor Lenz, Girish Dwivedi, Mohammed Bennamoun, Doo Seok Jeong, and Wei D. Lu. "Training Spiking Neural Networks Using Lessons From Deep Learning". *Proceedings of the IEEE*, 111(9), September 2023.